# Fundamentals 14 - Multi AgenticSystem Completo

Sistema completo multi-agente: solver, judge y reviewer LM opcional. Incluye pipeline, graph local, environment, eval y lineage.


In [ ]:
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=6, max_turns=6)
local_runtime = toolkit.runtime(provider="python-runtime", model="python-runtime", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
system = toolkit.AgenticSystem(model=lm_runtime.model_id or "python-runtime", region=lm_runtime.region_name or "local", runtime=lm_runtime)
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
toolkit.show({"runtime_auto": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


In [ ]:
@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}
@toolkit.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}


## Parámetros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


## 1) Agentes del sistema


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solver = toolkit.agent(name="multi_solver", instructions="Resuelve n?meros estructurados.", tools=[solve_arithmetic], engine="python-runtime", runtime=local_runtime, contract=toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic")), policy=policy)
judge = toolkit.agent(name="multi_judge", instructions="Valida resultado.", tools=[judge_result], engine="python-runtime", runtime=local_runtime, contract=toolkit.AgentContract(must_call=["judge_result"], tool_expectation=toolkit.expect.exactly("judge_result")), policy=policy)
reviewer = system.agent(name="multi_lm_reviewer", instructions="Revisa evidencia final sin cambiar n?meros.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))
toolkit.show({"agents": [solver.info(), judge.info(), reviewer.info()]})


## 2) Pipeline multi-agente + lineage


In [ ]:
solve = solver.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
judgement = judge.run({"tool": "judge_result", "input": {"result": solve.data["result"], "expected": EXPECTED}})
review = None
if lm_available:
    review = reviewer.run(str({"solution": solve.data, "judge": judgement.data}))
else:
    toolkit.show({"status": "skipped", "reason": lm_resolution["reason"]}, title="LM reviewer saltado")

final = {
    "procedimiento": solve.data["procedure"],
    "resultado_final": solve.data["result"],
    "judge": judgement.data,
    "lm_review": review.text if review else None,
}
result = toolkit.compose_result(
    text="Multi AgenticSystem completo ejecutado.",
    data=final,
    results=[solve, judgement, review],
    mode="multi-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = result.lineage(name="fundamentals.multi_agentic_system_full", question=USER_PROMPT, goal="Explicar pipeline multi-agente completo.")
toolkit.human_result(result, title="Human result - Multi AgenticSystem", pretty=PRETTY, show_lineage=True, lineage=lineage)

## 3) Graph local + environment + eval


In [ ]:
def node_solve(state: dict) -> dict:
    out = solver.run({"tool": "solve_arithmetic", "input": {"numbers": state["numbers"]}}).data
    return {**state, "procedure": out["procedure"], "result": out["result"]}
def node_judge(state: dict) -> dict:
    out = judge.run({"tool": "judge_result", "input": {"result": state["result"], "expected": EXPECTED}}).data
    return {**state, "judge": out}
graph_state = {"numbers": NUMBERS}
for node in [node_solve, node_judge]:
    graph_state = node(graph_state)
def transition_fn(row: dict, action: dict | None, info: dict) -> dict:
    solved = solver.run({"tool": "solve_arithmetic", "input": {"numbers": row["numbers"]}}).data
    judged = judge.run({"tool": "judge_result", "input": {"result": solved["result"], "expected": row["expected"]}}).data
    return {"result": solved["result"], "judge": judged}
def reward_fn(state: dict) -> float:
    return 1.0 if (state.get("judge") or {}).get("ok") else 0.0
env_records = [{"numbers": NUMBERS, "expected": EXPECTED}]
env = toolkit.AgenticEnvironment(name="multi_system_env", records=env_records, initial_memory={}, transition_fn=transition_fn, reward_fn=reward_fn)
env.reset()
_, reward, terminated, truncated, info = env.step()
report = toolkit.run_eval(solver, [{"id": "default", "input": {"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}}, "expected": {"result": EXPECTED}}])
toolkit.show({"graph_state": graph_state, "env_summary": env.summary(), "step": info["transition"], "eval_report": report.to_dict()})


In [ ]:
toolkit.show({"notebook": "14_multi_agentic_system_api.ipynb", "api_coverage": ["multi-agent pipeline", "runtime(provider='auto')", "graph nodes", "AgenticEnvironment", "run_eval", "compose_result", "RunResult.lineage"]})


## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `AgenticSystem`: Ciclo completo multi-agent.
- `Tool -> Agents -> System -> Graph -> Environment -> Eval`: Ruta completa multi-agente.
- `run_eval`: Validacion empirica del sistema.
- `human_result`: Presentacion humana final.

